# Seasonal hotspots in satellite composites

This notebook takes a stack of gridded composites (one CSV per acquisition, lattice of
longitude, latitude, value), bins them to a hexagonal grid, builds seasonal covariates per hex,
and screens each covariate for spatial clusters with Getis-Ord Gi* and local Moran's I. The
outputs are a hex GeoPackage with hotspot classes, QML styles, and a Kepler.gl map with one layer
per covariate.

The composites are synthetic chlorophyll-a fields: four years, six days of year each, on a 60 by
60 degree lattice over the North Atlantic, with a seasonal cycle, a coastal gradient, a slow
interannual trend and 8 percent cloud gaps. The reader only needs the acquisition date in the
filename (`AYYYYDDD` by default), so real composites drop in without code changes.

Dask is used for the point to hex join. The cluster dashboard link is printed when the client
starts; on WSL the `localhost` form opens in the Windows browser. `DASK_N_WORKERS`,
`DASK_THREADS_PER_WORKER`, `DASK_MEMORY_LIMIT` and `DASK_DASHBOARD_ADDRESS` override the
defaults; `DASK_OPEN_DASHBOARD=1` opens the page as the client starts.

In [ ]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
warnings.filterwarnings("ignore")

DATA = ROOT / "data" / "oceanography"
EXPORTS = ROOT / "exports" / "oceanography"
DATA.mkdir(parents=True, exist_ok=True)
EXPORTS.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
print("repo root:", ROOT)

In [ ]:
from datasets.synthetic import synthetic_chlorophyll_grids
from fortress_gis.compute.cluster import get_dask_client
from fortress_gis.domains import oceanography as oc
from fortress_gis.features.seasonal import missingness_report
from fortress_gis.stats.hypothesis import benjamini_hochberg
from fortress_gis.viz.kepler import KeplerMapBuilder, kepler_available

paths = sorted(DATA.glob("A*_chlor_a.csv"))
if not paths:
    paths = synthetic_chlorophyll_grids(
        DATA,
        years=(2016, 2017, 2018, 2019),
        days_of_year=(15, 60, 135, 200, 250, 320),
        lon=(-40.0, -10.0, 60),
        lat=(25.0, 55.0, 60),
    )
print(len(paths), "acquisitions, first:", paths[0].name)

In [ ]:
client = get_dask_client()  # prints the dashboard link; DASK_N_WORKERS etc. override defaults
client

## Read the composites as one point layer

`load_composites` reads every CSV and joins them on the lattice coordinates, giving one point
per lattice cell and one column per acquisition named `chlor_<year>_<doy>`. The join is on
rounded coordinates, so grids that differ by floating point noise still line up.

In [ ]:
points = oc.load_composites(paths)
print(points.shape)
points.iloc[:3, :6]

## Bin to hexes and build seasonal covariates

`build_hex_covariates` lays a hex grid over the points (radius three times the lattice
spacing unless given), joins points to hexes with `dask_geopandas.sjoin`, and averages each
acquisition per hex. Acquisitions are then labelled by season from the day of year. The default
`SeasonCalendar` is a two season split, winter for day of year below 104 or from 288 and summer
between; `SeasonCalendar.four_seasons()` gives meteorological seasons. Per hex the stack holds:

- `<season>_mean`: mean over years of that season's mean
- `total_mean`: mean over every acquisition
- `<season>_var`: variance across years of the year over year change in that season
- `<year>_var`: variance across seasons of the year over year change within that year

`missingness_report` on the point table gives the fraction of lattice cells with no value per
acquisition, which is where cloud gaps and swath edges show up. Hex means absorb most of it.

In [ ]:
stack = oc.build_hex_covariates(points, use_dask=True)
print("hex cells:", len(stack.cells), " covariates:", len(stack.covariate_columns))
print(stack.covariate_columns)
missing = missingness_report(points.drop(columns="geometry"))
missing[missing.index.str.startswith("chlor_")].sort_values(ascending=False).head(6).round(3)

In [ ]:
stack.covariates.drop(columns="geometry").describe().T.round(3)

## Screen for spatial clusters

`screen_hotspots` runs one local statistic per covariate over a distance band weights matrix
(threshold set so every hex has at least one neighbour). Gi* returns a z-score and a
permutation p-value per hex and classes hexes as hot or cold at 90, 95 and 99 percent
confidence. Local Moran's I classes hexes as HH, LL, HL, LH or not significant. Both use 499
permutations here; 999 is the usual choice for a report and takes about twice as long.

In [ ]:
gstar = oc.screen_hotspots(
    stack.covariates,
    ["winter_mean", "summer_mean", "summer_var", "total_mean"],
    method="gstar",
    permutations=499,
)
for name, res in gstar.items():
    print(name)
    print(res.summary().to_string(), "\n")

In [ ]:
lisa = oc.screen_hotspots(stack.covariates, ["2018_var"], method="lisa", permutations=499)
lisa["2018_var"].summary()

## Correct for multiple testing

Every hex gets its own test, so at the 5 percent level 5 percent of hexes are flagged by chance
alone. `benjamini_hochberg` adjusts the permutation p-values to control the false discovery rate
and reports how many survive.

In [ ]:
layer = gstar["summer_mean"].layer
bh = benjamini_hochberg(layer["summer_mean_gi_p_sim"], alpha=0.05)
print("raw p < 0.05:", int((layer["summer_mean_gi_p_sim"] < 0.05).sum()))
print("BH significant:", int(bh["significant"].sum()), "of", len(bh))

In [ ]:
merged = oc.merge_hotspot_layers({**gstar, **lisa})
class_cols = [c for c in merged.columns if c.endswith("_class")]
merged[["hex_id", *class_cols]].head()

## Kepler.gl map

The cell below renders the map inline. If the widget shows as blank text, run
`jupyter nbextension enable --py --sys-prefix keplergl` once in the environment and reload the
page; `fortress_gis.viz.kepler.enable_nbextension()` prints the same command. The HTML export in
the last section does not need the extension and opens in any browser.

In [ ]:
from fortress_gis.viz.kepler import HOTSPOT_CLASS_COLORS

builder = KeplerMapBuilder(title="Seasonal hotspots", height=550)
builder.add_layer(stack.covariates, "summer mean", color_field="summer_mean", opacity=0.6)
builder.add_layer(
    gstar["summer_mean"].layer,
    "summer mean Gi*",
    color_field="summer_mean_gi_class",
    categorical_colors=HOTSPOT_CLASS_COLORS,
    opacity=0.7,
)
builder.widget() if kepler_available() else print("keplergl not installed")

## Export

The QGIS bundle holds the hex covariates and one layer per screened attribute with a
categorized QML style matching the Kepler colours. The Kepler HTML has every layer with the
covariate layers hidden by default.

In [ ]:
paths_out = oc.export_artifacts(stack, {**gstar, **lisa}, EXPORTS, name="composites")
for k, v in paths_out.items():
    print(f"{k:>10}: {v.relative_to(ROOT)}")
client.close()